# AirBnB NYC Analytics Project

* Project by Nhi Bui · Villanova University · [GitHub](https://github.com/nhibui23/airbnb-nyc-product-analytics-project) · [LinkedIn](https://linkedin.com/in/nhiuyenbui)

# 05. Review Count Curve

> "At what point do additional reviews stop influencing bookings?"

Common host advice is to "get more reviews." But if a listing already has 500 reviews and a 5-star rating, does going to 1,500 reviews meaningfully change how often it books? In this notebook, I will test whether review count has a diminishing returns effect on booking activity.

→ **Interpretation** This will change what hosts should focus on. 

For example: a host with 20 reviews should try to grow more, while a host with 500 reviews may be better off focusing on positioning, pricing, and event theming,  which is what the Airbnb AI prototype recommends. This gives prototype the basis for that distinction.

**My Approach:**
1. Filter to highly-rated listings (rating ≥ 4.5) so review count is the variable of interest
2. Bin listings into 5 review-count buckets
3. Compare average occupancy across bins
4. Use a Welch's t-test between adjacent bins to identify where the effect flattens

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/Airbnb_Open_Data_Cleaned.csv')

# Rebuild the occupancy proxy (same definition as Notebook 04)
df['occupancy_proxy'] = (365 - df['availability 365']) / 365

# Apply Airbnb branding
airbnb_coral = '#FF5A5F'
airbnb_teal = '#00A699'
airbnb_orange = '#FC642D'
airbnb_dark = '#484848'
airbnb_gray = '#767676'

airbnb_palette = [airbnb_coral, airbnb_teal, airbnb_orange, airbnb_dark, airbnb_gray]
sns.set_palette(airbnb_palette)
sns.set_style('whitegrid')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.family'] = 'DejaVu Sans'

print(f"Dataset: {df.shape[0]:,} listings")

Dataset: 63,718 listings


# 1. Filter to highly-rated listings

To see the effect of review count, we filter to listings with rating ≥ 4.5 stars (high quality). This holds listing quality constant, so any pattern we see in the data reflects the impact of review count itself, not the fact that higher-quality listings tend to get more reviews and more bookings for reasons unrelated to review count.

→ We need to see the impact of number of reviews of review rate number to actually determine the tradeoff point of number of reviews

In [2]:
# Filter to highly-rated listings
high_rated = df[df['review rate number'] >= 4.5].copy()

print(f"Full dataset: {len(df):,} listings")
print(f"High-rated (≥ 4.5 stars): {len(high_rated):,} listings")
print(f"Share of total: {len(high_rated) / len(df) * 100:.1f}%")

Full dataset: 63,718 listings
High-rated (≥ 4.5 stars): 14,680 listings
Share of total: 23.0%


# 2. Bin listings by review count

Listings are split into 5 buckets based on number of reviews:

| Bucket | Review count |
|---|---|
| Very Low | 0-10 |
| Low | 11-50 |
| Medium | 51-200 |
| High | 201-500 |
| Very High | 500+ |

Because a linear split would put nearly everything in the first bucket for these right-skew review counts, these bins are each roughly 3-5x the previous 

In [3]:
# Bin by review count
high_rated['review_count_bin'] = pd.cut(high_rated['number of reviews'],
                                        bins=[-1, 10, 50, 200, 500, high_rated['number of reviews'].max()],
                                        labels=['Very Low (0-10)', 'Low (11-50)', 'Medium (51-200)', 'High (201-500)', 'Very High (500+)'])


In [4]:
# Summary stats per bin
summary = high_rated.groupby('review_count_bin').agg(
    listings=('number of reviews', 'count'),
    avg_reviews=('number of reviews', 'mean'),
    avg_occupancy=('occupancy_proxy', 'mean'),
    avg_reviews_per_month=('reviews per month', 'mean')
).round(3)

print(summary)

                  listings  avg_reviews  avg_occupancy  avg_reviews_per_month
review_count_bin                                                             
Very Low (0-10)       6205        4.077          0.505                  0.726
Low (11-50)           5041       25.135          0.552                  1.764
Medium (51-200)       3113       94.852          0.536                  2.933
High (201-500)         312      276.381          0.515                  4.790
Very High (500+)         9      556.667          0.311                 13.563


### Key Takeaway

* We see 2 patterns from the output

**1. Occupancy shows no clear review-count effect.** 

* Contrasting to our previous hypothesis, occupancy is flat across review count bins, ranging from 50.5% (Very Low) to 55.2% (Low), then falling back to 51.5% (High)

* The Very High bin holds only 9 listings and is not reliable

* If reviews had diminishing returns on occupancy, we would expect to see occupancy rise steeply from Very Low to Low, then flatten, but the data does not show this pattern.

**2. Reviews per month rises with review count.** 

* Surprisingly, reviews per month rises from 0.73 at Very Low to 4.79 at High

* This pattern suggests review count tracks booking activity when measured by reviews per month, but not by the occupancy proxy, as listings with more reviews accumulate them faster because they book more frequently

→ **Limitations** Only 9 listings in this dataset have > 500 reviews. The dataset lacks the high-review-count listings needed to identify a flattening point at that scale

→ **Question for the t-test:** In the next step, we will test whether the small differences we see between bins are statistically significant, or if occupancy is truly flat across review count

In [5]:
# Adjacent-pair t-tests
bin_order = ['Very Low (0-10)', 'Low (11-50)', 'Medium (51-200)', 'High (201-500)', 'Very High (500+)']

print(f"{'Comparison':<40} {'p-value':>12} {'Significant?':>15}")
print("-" * 70)

for i in range(len(bin_order) - 1):
    group_a = high_rated[high_rated['review_count_bin'] == bin_order[i]]['occupancy_proxy']
    group_b = high_rated[high_rated['review_count_bin'] == bin_order[i+1]]['occupancy_proxy']
    
    t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)
    significant = "Yes" if p_value < 0.05 else "No"
    
    label = f"{bin_order[i]} vs {bin_order[i+1]}"
    print(f"{label:<40} {p_value:>12.4f} {significant:>15}")

Comparison                                    p-value    Significant?
----------------------------------------------------------------------
Very Low (0-10) vs Low (11-50)                 0.0000             Yes
Low (11-50) vs Medium (51-200)                 0.0264             Yes
Medium (51-200) vs High (201-500)              0.2522              No
High (201-500) vs Very High (500+)             0.1062              No


### Key Takeaway

* 2 of the 4 adjacent-pair comparisons are statistically significant.

* Occupancy rises significantly from Very Low to Low (50.5% to 55.2%), then drops slightly from Low to Medium (55.2% to 53.6%), and flattens from Medium onward. The Very High bin (500+) holds only 9 listings and is not reliable.

→ **Interpretation** 

* Getting from 0 to 50 reviews has a measurable effect on booking activity

* Past 50 reviews, additional reviews do not improve occupancy and may slightly reduce it, possibly because high-review listings tend to be older or in more competitive neighborhoods

→ Hosts with fewer than 50 reviews should focus on building their review count, while hosts with 50 or more reviews are past the point where more reviews help, which Airbnb AI prototype can direct these hosts toward positioning, pricing, and event-based theming instead.

## Summary

| Analysis | Finding |
|---|---|
| High-rated listings filter | 14,680 listings with rating ≥ 4.5 |
| Occupancy by review bin | Peaks at 55.2% in the 11-50 review bin, then flat or declining |
| Flattening point | ~50 reviews, so the first non-significant t-test is Medium vs High (p = 0.25) |
| Limitations | Only 9 listings have 500+ reviews, so the 500–1,000 may not be testable in this dataset |


* While occupancy flattens after 50 reviews, reviews per month rises steadily across all bins (0.73 at Very Low to 4.79 at High). This is likely a mechanical relationship, as listings that have been active longer accumulate more total reviews and  receive more reviews per month simply because they have more booking history

→ **Proposed approach:** We can conduct an A/B test to isolate the causal question by boosting some low-review listings in search rankings and measure whether the resulting increase in reviews leads to improved occupancy improvement, or whether occupancy returns to normal again once we end the boost.


## Handoff

These findings support 2 sources:

**Notebook 06:** The 50-review threshold is a specific cutoff that segments hosts into 2 groups: "chase reviews" vs "focus on positioning." 

* We will combine this with the guest-side and host-side findings into the final recommendation and propos an A/B test to validate the threshold on live data.

**Airbnb AI prototype** The prototype uses the 50-review threshold to decide which type of recommendation to give. Below 50 reviews, it prompts hosts toward review generation strategies, while above 50, it moves to positioning and event-based recommendations.